# Modelo final

Projeto **Ignis**, parceria Whirlpool e Inteli. Este notebook apresenta o modelo que o grupo
indica para uso em produção, e o executa de ponta a ponta: ajuste, medição na validação
cruzada, avaliação no conjunto de teste reservado e um exemplo de triagem.

O código do modelo é o mesmo da **Seção 23 de `notebooks/colabs_S4/modelagem.ipynb`**,
reproduzido aqui sem alteração. Não é uma reescrita: é a mesma seção que
`src/frontend/dados/exportar.py` executa para gerar os modelos que o dashboard consulta, de modo
que o que se vê aqui é o que roda em produção.

O percurso completo que levou a esta formulação, com as alternativas medidas e descartadas,
está documentado em `notebooks/modelo_final/guia_modelo_final.ipynb`.

## Como executar

**No Colab.** Rode a célula seguinte e envie os nove arquivos quando solicitado: os sete
`dataset_<area>_13cat.csv`, o `limites_ativos.csv` e o `particao_treino_teste.json`. Os sete
primeiros e o oitavo são gerados pelos notebooks `preparacao_datasets` e
`tratamento_datasets_13cat`; o nono está versionado no repositório, em
`notebooks/colabs_S3/preparacao_testes/`. Os dados não acompanham o repositório por
compromisso de proteção de dados com a parceira.

**No repositório.** A mesma célula detecta que já está no projeto e apenas entra na pasta
correta, sem pedir envio.

A execução leva poucos minutos: são sete áreas, dezessete dobras e três sementes.

In [1]:
# Preparação do ambiente. Esta célula não faz parte do modelo: ela apenas organiza os
# arquivos no formato que a Seção 23 espera, isto é, os CSV na pasta de trabalho e a
# partição em ../colabs_S3/preparacao_testes/. Assim a célula seguinte é cópia fiel do
# notebook de modelagem, sem adaptação de caminho.
import os
import pathlib
import shutil

AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling",
         "mainback", "ipc_comp", "coil_temps"]
CSVS = [f"dataset_{a}_13cat.csv" for a in AREAS] + ["limites_ativos.csv"]
PARTICAO = "particao_treino_teste.json"


def raiz_repositorio(inicio=None):
    """Raiz do projeto, se o notebook estiver rodando dentro dele."""
    atual = pathlib.Path(inicio or pathlib.Path.cwd()).resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / "notebooks").is_dir() and (pasta / "documents").is_dir():
            return pasta
    return None


RAIZ = raiz_repositorio()
if RAIZ is not None:
    os.chdir(RAIZ / "notebooks" / "colabs_S4")
    print("Rodando dentro do repositório:", os.getcwd())
else:
    # Colab: recebe os arquivos e monta a mesma estrutura de pastas.
    base = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
    trabalho, preparacao = base / "colabs_S4", base / "colabs_S3" / "preparacao_testes"
    trabalho.mkdir(parents=True, exist_ok=True)
    preparacao.mkdir(parents=True, exist_ok=True)
    # Um arquivo por vez: a versao anterior testava a particao no lugar de cada CSV,
    # e por isso nao pedia envio quando so a particao ja estava na pasta.
    faltando = [f for f in CSVS if not (trabalho / f).exists()]
    if not (preparacao / PARTICAO).exists():
        faltando.append(PARTICAO)
    if faltando:
        try:
            from google.colab import files
            print("Envie os", len(CSVS) + 1, "arquivos:", ", ".join(CSVS + [PARTICAO]))
            enviados = files.upload()
        except ImportError:
            enviados = {}
        for nome in list(enviados) or [p.name for p in base.glob("*.csv")] + [PARTICAO]:
            orig = base / nome
            if not orig.exists():
                continue
            shutil.move(str(orig), str((preparacao if nome == PARTICAO else trabalho) / nome))
    os.chdir(trabalho)
    print("Pasta de trabalho:", os.getcwd())

ausentes = [f for f in CSVS if not pathlib.Path(f).exists()]
ausentes += [] if pathlib.Path("../colabs_S3/preparacao_testes/" + PARTICAO).exists() else [PARTICAO]
if ausentes:
    raise SystemExit("faltam arquivos: " + ", ".join(ausentes))
print("Os nove arquivos estão no lugar.")

Rodando dentro do repositório: C:\Users\Inteli\Desktop\g01\notebooks\colabs_S4
Os nove arquivos estão no lugar.


Os nove arquivos estão no lugar, e a saída informa a pasta de
trabalho adotada. Quando o notebook roda dentro do repositório, como acima, a célula apenas
entra em `notebooks/colabs_S4`; fora dele, ela recebe os arquivos e monta a mesma estrutura.
Nos dois casos o que vem a seguir enxerga os dados no mesmo lugar, e por isso a próxima célula
pôde ser copiada do notebook de modelagem sem adaptação de caminho.

## A formulação entregue

O modelo final **não é um estimador só**: é um por área térmica, e a escolha do estimador e do
tratamento de `losses` muda conforme a área.

| Área | Estimador | `losses` |
|---|---|---|
| `oven_hmi` | Random Forest (BREIMAN, 2001) | agrupadas em quatro níveis |
| `ct_knobs` | Random Forest | treze categorias |
| `main_vent_sys` | média entre Extra Trees (GEURTS; ERNST; WEHENKEL, 2006) e Gradient Boosting | treze categorias |
| `ipc_cooling` | Random Forest | agrupadas em quatro níveis |
| `mainback` | Random Forest | treze categorias |
| `ipc_comp` | Random Forest | agrupadas em quatro níveis |
| `coil_temps` | média entre Random Forest e Gradient Boosting (FRIEDMAN, 2001) | treze categorias |

Todos com trezentas árvores e os demais hiperparâmetros no padrão do scikit-learn
(PEDREGOSA et al., 2011), semente 42.

Três decisões completam a receita. O treino usa o **formato longo**, com uma linha por par de
ensaio e componente, e a identidade do componente é substituída pelo **patamar** e pela
**escala** das suas temperaturas, calculados apenas nas linhas de treino de cada dobra, o que
impede o vazamento do alvo por construção (KAUFMAN et al., 2012). O ajuste recebe um **peso
próximo ao limite**, que dá peso três às leituras sobre o limite do componente e praticamente
um às distantes trinta graus dele, porque é ali que a triagem decide. E os valores ausentes do
alvo são **descartados por máscara**, nunca imputados.

A célula seguinte é a definição dessas funções, copiada da Seção 23 sem alteração.

In [2]:
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler

_PART = json.load(open("../colabs_S3/preparacao_testes/particao_treino_teste.json", encoding="utf-8"))
_DOBRA = {int(k): v for k, v in _PART["dobra_por_run_idx"].items()}
_TESTE = set(_PART["run_idx_teste"])
_LIM = pd.read_csv("limites_ativos.csv").set_index("tc_name")["limite_num"]
_PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
         "deflector_hole_", "material_coil_support_", "reason_", "loss_profile_order_")
_ORDEM = "loss_profile_order_"
_PERDAS = "losses_"
_AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]
_SEMENTES = (42, 1, 2)
_ESTIMADORES = ("rf", "rf+gb")
_TRATAMENTOS = ("detalhado", "agrupado")
_CANDIDATOS = [(e, t) for e in _ESTIMADORES for t in _TRATAMENTOS]
# Main Vent Sys recebe dois candidatos a mais: a media entre Extra Trees e gradiente,
# com as perdas detalhadas e agrupadas. A mescla foi identificada depois da comparacao
# de algoritmos da quarta sprint, em que venceu a entrega nessa regiao em todas as
# sementes e variantes; foi acrescentada so aqui, e nao em todas as regioes, porque a
# grade ampla medida com aninhamento estrito piorou o conjunto quando todas as regioes
# escolhiam entre vinte candidatos, e porque em Coil Temps a mesma mescla reduz o erro
# medio mas dobra as aprovacoes indevidas.
_EXTRA_POR_AREA = {"main_vent_sys": [("et+gb", "detalhado"), ("et+gb", "agrupado")]}
_CANDIDATOS_DE = {a: _CANDIDATOS + _EXTRA_POR_AREA.get(a, []) for a in _AREAS}
_MARGEM = 8
_PESO = (317 - 44) / (74 - 25)   # razao de custo revelada pela escolha da margem


def _familia_de_perdas(coluna):
    """O perfil dissipado, sem a ordem das bobinas e sem o tipo de ensaio.

    A ordem das bobinas ja e coluna propria e o tipo de ensaio ja e a linha de
    run case, de modo que o que resta de informacao nova nesta coluna e qual
    perfil de potencia foi imposto.
    """
    t = coluna[len(_PERDAS):].lower()
    if "no losses" in t:
        return "none"
    if "simone" in t:
        return "simone"
    if "miro" in t:
        return "miro"
    return "other"


def _bloco_agrupado(d, colunas_perdas):
    """Niveis mutuamente exclusivos no lugar dos indicadores de perdas.

    O nivel de nao declarado e derivado da ausencia de qualquer indicador, e nao
    de um rotulo proprio, porque a planilha deixa a celula vazia nesses ensaios.
    """
    fam = np.array([_familia_de_perdas(c) for c in colunas_perdas])
    A = d[colunas_perdas].to_numpy(float)
    niveis = sorted(set(fam.tolist()) | {"not_declared"})
    M = np.zeros((len(d), len(niveis)))
    for j, nivel in enumerate(niveis):
        if nivel == "not_declared":
            M[:, j] = (A.sum(axis=1) == 0).astype(float)
        else:
            M[:, j] = (A[:, fam == nivel].sum(axis=1) > 0).astype(float)
    assert (M.sum(axis=1) == 1).all(), "os niveis de perdas deixaram de ser exclusivos"
    return M, niveis


def _entradas(area, tratamento):
    """Matriz de entrada da regiao, sem as colunas de ordem de chaveamento, que
    foram medidas como prejudiciais."""
    d = pd.read_csv(f"dataset_{area}_13cat.csv")
    cols = [c for c in d.columns if c.startswith(_PREF) and not c.startswith(_ORDEM)]
    perdas = [c for c in cols if c.startswith(_PERDAS)]
    assert perdas, f"{area}: bloco de perdas ausente"
    if tratamento == "detalhado":
        X = d[cols].to_numpy(float)
    else:
        outras = [c for c in cols if c not in perdas]
        M, _ = _bloco_agrupado(d, perdas)
        X = np.hstack([d[outras].to_numpy(float), M])
    alvos = [c for c in d.columns if not c.startswith(_PREF) and d[c].notna().sum() >= 5]
    # Toda coluna tratada como alvo precisa ser um termopar da tabela de limites. Sem
    # esta verificacao, uma coluna de configuracao cujo prefixo nao estivesse em _PREF
    # seria classificada como alvo e prevista como se fosse temperatura, sem erro
    # algum. Foi o que aconteceu com o motivo do ensaio quando mudou de nome de cfg_
    # para reason_, primeiro neste caderno e depois no de busca aleatoria.
    _estranhas = [c for c in alvos if c not in _LIM.index]
    assert not _estranhas, (f"{area}: colunas tratadas como alvo que nao sao termopares: "
                            f"{_estranhas}. Declare o prefixo em _PREF.")
    return d, X, alvos


def _longo(d, X, alvos):
    Y = d[alvos].to_numpy(float)
    Xl, yl, rl, cl = [], [], [], []
    for j in range(Y.shape[1]):
        ok = ~np.isnan(Y[:, j])
        Xl.append(X[ok]); yl.append(Y[ok, j])
        rl.append(np.arange(len(d))[ok]); cl.append(np.full(ok.sum(), j))
    Xl, yl = np.vstack(Xl), np.concatenate(yl)
    rl, cl = np.concatenate(rl), np.concatenate(cl)
    return Xl, yl, cl, rl, [alvos[j] for j in cl]


def _carrega(area, tratamento):
    """Formato longo da regiao, sem o conjunto de teste reservado."""
    d, X, alvos = _entradas(area, tratamento)
    Xl, yl, cl, rl, comps = _longo(d, X, alvos)
    gl = np.array([_DOBRA.get(int(r)) for r in rl], dtype=object)
    ok = np.array([g is not None for g in gl]) & ~np.isin(rl, list(_TESTE))
    return (Xl[ok], yl[ok], cl[ok], gl[ok].astype(int), rl[ok],
            [c for c, m in zip(comps, ok) if m])


def _peso_perto_do_limite(y, comps):
    """Peso maior para as leituras proximas do limite do componente.

    A triagem decide comparando a previsao mais a margem contra o limite, de modo
    que o erro cometido longe do limite nao muda decisao alguma. Ponderar o ajuste
    pela proximidade ao limite concentra a capacidade do modelo onde a decisao
    acontece.

    A forma foi fixada antes de medir e nao foi ajustada depois: peso um mais dois
    vezes uma gaussiana de largura quinze graus, de modo que uma leitura sobre o
    limite pesa tres vezes e uma a trinta graus dele pesa praticamente um.
    Componentes sem limite tabelado recebem peso um.
    """
    lim = np.array([_LIM.get(c, np.nan) for c in comps], float)
    d = np.abs(y - lim)
    return np.where(np.isnan(lim), 1.0, 1.0 + 2.0 * np.exp(-(np.nan_to_num(d, nan=1e9) / 15.0) ** 2))


def _ajusta(Xt, yt, Xv, estimador, semente, peso=None):
    """rf: floresta; rf+gb e et+gb: media da floresta ou do Extra Trees com o gradiente."""
    base = ExtraTreesRegressor if estimador.startswith("et") else RandomForestRegressor
    p = base(n_estimators=300, random_state=semente, n_jobs=-1).fit(
        Xt, yt, sample_weight=peso).predict(Xv)
    if estimador.endswith("+gb"):
        p = (p + GradientBoostingRegressor(n_estimators=300, random_state=semente).fit(
            Xt, yt, sample_weight=peso).predict(Xv)) / 2
    return p


def _fora_da_dobra(area, candidato, semente):
    """Erro de cada leitura quando a sua configuracao esta fora do treino."""
    estimador, tratamento = candidato
    Xl, yl, cl, gl, rl, comps = _carrega(area, tratamento)
    saida = {}
    for k in sorted(set(gl)):
        va, tr = gl == k, gl != k
        if tr.sum() < 30 or not va.any():
            continue
        pat = pd.Series(yl[tr]).groupby(cl[tr]).mean()
        dsv = pd.Series(yl[tr]).groupby(cl[tr]).std().fillna(1.0).replace(0, 1.0)
        ger = float(yl[tr].mean())
        M = lambda m: np.hstack([Xl[m],
                                 pd.Series(cl[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                                 pd.Series(cl[m]).map(dsv).fillna(1.0).to_numpy().reshape(-1, 1)])
        s = StandardScaler()
        Xt, Xv = s.fit_transform(M(tr)), s.transform(M(va))
        p = _ajusta(Xt, yl[tr], Xv, estimador, semente,
                    _peso_perto_do_limite(yl[tr], [comps[i] for i in np.where(tr)[0]]))
        for idx, i in enumerate(np.where(va)[0]):
            saida[(int(rl[i]), comps[i])] = (p[idx] - yl[i], yl[i], comps[i])
    return saida


_ndet = len(_entradas("mainback", "detalhado")[1][0])
_nagr = len(_entradas("mainback", "agrupado")[1][0])
_d0 = pd.read_csv("dataset_mainback_13cat.csv")
_, _niveis = _bloco_agrupado(_d0, [c for c in _d0.columns if c.startswith(_PERDAS)])
print(f"Niveis do tratamento agrupado: {_niveis}")
print(f"Colunas de entrada: {_ndet} no tratamento detalhado, {_nagr} no agrupado")
print()

_erro = {}
for _s in _SEMENTES:
    for _a in _AREAS:
        for _c in _CANDIDATOS_DE[_a]:
            _erro.setdefault((_c, _s), {})[_a] = _fora_da_dobra(_a, _c, _s)
    print(f"  semente {_s} concluida", flush=True)

Niveis do tratamento agrupado: ['miro', 'none', 'not_declared', 'simone']
Colunas de entrada: 33 no tratamento detalhado, 24 no agrupado

  semente 42 concluida
  semente 1 concluida
  semente 2 concluida


As funções estão definidas e a saída confirma duas decisões da
formulação.

O tratamento agrupado reduz as treze categorias de `losses` a quatro níveis: `miro`, `none`,
`simone` e `not_declared`, este último para os ensaios em que a planilha deixa a célula vazia.

A contagem de colunas mostra o efeito disso na entrada do modelo. São **33 colunas no
tratamento detalhado**, e não as 37 que os arquivos trazem, porque as quatro colunas de ordem
de chaveamento dos IGBTs ficam deliberadamente de fora, por terem sido medidas como
prejudiciais. No tratamento agrupado são **24**: as mesmas 33 menos as treze categorias
detalhadas, mais os quatro níveis agrupados.

### Medição na validação cruzada

A validação retira **uma configuração experimental inteira por vez**, com todos os seus
componentes e repetições, em dezessete dobras. Uma divisão aleatória das linhas colocaria
componentes do mesmo ensaio nos dois lados, e o modelo passaria a reconhecer a configuração em
vez de generalizar para uma nova.

A célula compara a formulação entregue com a anterior, que escolhia apenas o estimador, e
refaz a escolha dentro de cada dobra para que o número já inclua o custo de escolher.

A escolha é refeita pelo **método aproximado**, que é levemente otimista. O resultado, 5,961
graus, é portanto o limite inferior da faixa que o projeto reporta; o limite superior, 6,23,
vem do método estrito, medido na seção 1 de
`notebooks/colabs_S4/analises_complementares.ipynb`.

In [3]:
# A formulacao anterior, usada como referencia do ganho, e a escolha de estimador
# por regiao com as perdas detalhadas, que foi a entrega antes desta auditoria.
_ANTERIOR = {"oven_hmi": "rf", "ct_knobs": "rf", "main_vent_sys": "rf+gb",
             "ipc_cooling": "rf", "mainback": "rf", "ipc_comp": "rf+gb", "coil_temps": "rf+gb"}

_res = {n: {} for n in ("anterior", "escolhido")}
_escolhas = {a: [] for a in _AREAS}
for _s in _SEMENTES:
    for area in _AREAS:
        ref = _erro[((_ANTERIOR[area], "detalhado"), _s)][area]
        for d in sorted({_DOBRA[ch[0]] for ch in ref}):
            cand = {c: float(np.mean([abs(v[0]) for ch, v in _erro[(c, _s)][area].items()
                                      if _DOBRA[ch[0]] != d])) for c in _CANDIDATOS_DE[area]}
            melhor = min(cand, key=cand.get)
            _escolhas[area].append(melhor)
            for ch in ref:
                if _DOBRA[ch[0]] == d:
                    _res["anterior"].setdefault(_s, {})[(area,) + ch] = ref[ch]
                    _res["escolhido"].setdefault(_s, {})[(area,) + ch] = _erro[(melhor, _s)][area][ch]

print("Candidato escolhido por regiao, nas dobras e sementes:\n")
_DOMINANTE = {}
for area in _AREAS:
    c = Counter(_escolhas[area])
    _DOMINANTE[area] = c.most_common(1)[0][0]
    print(f"  {area:<15}" + "  ;  ".join(f"{e}|{t}: {n}/{len(_escolhas[area])}"
                                         for (e, t), n in c.most_common()))
print("\nRegra dominante, que e a que a entrega adota:")
for a, (e, t) in _DOMINANTE.items():
    print(f"  {a:<15} {e:<6} perdas {t}")

print("\nErro medio absoluto de cada candidato, por regiao, para leitura do porque:\n")
_chaves = {a: sorted(_erro[(_CANDIDATOS[0], _SEMENTES[0])][a]) for a in _AREAS}
_TODOS = _CANDIDATOS + [c for a in _AREAS for c in _EXTRA_POR_AREA.get(a, []) if c not in _CANDIDATOS]
print(f"{'regiao':<16}" + "".join(f"{e + ' ' + t:>22}" for e, t in _TODOS))
for area in _AREAS:
    linha = []
    for c in _TODOS:
        if c not in _CANDIDATOS_DE[area]:
            linha.append("-")
            continue
        v = np.mean([np.mean([abs(_erro[(c, s)][area][ch][0]) for ch in _chaves[area]])
                     for s in _SEMENTES])
        linha.append(f"{v:.3f}")
    print(f"{area:<16}" + "".join(f"{x:>22}" for x in linha))


def _metricas(mapa):
    dif = np.array([v[0] for v in mapa.values()], float)
    real = np.array([v[1] for v in mapa.values()], float)
    comps = [v[2] for v in mapa.values()]
    fa = fr = det = tot = 0
    for r, d, c in zip(real, dif, comps):
        lim = _LIM.get(c)
        if lim is None or pd.isna(lim):
            continue
        estoura, alarme = r >= lim, (r + d) + _MARGEM >= lim
        tot += estoura; fa += estoura and not alarme
        fr += (not estoura) and alarme; det += estoura and alarme
    return {"n": len(dif), "MAE": float(np.mean(np.abs(dif))),
            "RMSE": float(np.sqrt(np.mean(dif ** 2))),
            "R2": float(1 - (dif ** 2).sum() / ((real - real.mean()) ** 2).sum()),
            "vies": float(dif.mean()), "MAPE_%": float(np.mean(np.abs(dif) / real) * 100),
            "dentro_10_%": float(np.mean(np.abs(dif) / real < 0.10) * 100),
            "custo": float(np.mean(np.where(dif < 0, -dif * _PESO, dif))),
            "falsos_aprovados": fa, "falsos_reprovados": fr,
            "deteccao_%": (100.0 * det / tot if tot else float("nan"))}


_comuns = sorted(set.intersection(*[set(_res["anterior"][s]) for s in _SEMENTES]))
_a = np.mean([[abs(_res["anterior"][s][c][0]) for c in _comuns] for s in _SEMENTES], axis=0)
_b = np.mean([[abs(_res["escolhido"][s][c][0]) for c in _comuns] for s in _SEMENTES], axis=0)
_d = _a - _b
_bs = [np.random.default_rng(i).choice(_d, len(_d), replace=True).mean() for i in range(3000)]
print(f"\nMAE da formulacao anterior, estimador por regiao e perdas detalhadas: {_a.mean():.3f} C")
print(f"MAE da formulacao com estimador e tratamento das perdas por regiao:   {_b.mean():.3f} C")
print(f"ganho {_d.mean():+.3f} C | IC95 [{np.percentile(_bs, 2.5):+.3f}, {np.percentile(_bs, 97.5):+.3f}] "
      f"| wilcoxon p={stats.wilcoxon(_a, _b).pvalue:.2e}")

print("\nMetricas por regiao da formulacao adotada:\n")
_linhas = {}
for area in _AREAS + ["AGREGADO"]:
    por_s = []
    for s in _SEMENTES:
        m = {k: v for k, v in _res["escolhido"][s].items() if area == "AGREGADO" or k[0] == area}
        por_s.append(_metricas(m))
    _linhas[area] = {k: float(np.mean([p[k] for p in por_s])) for k in por_s[0]}
_T = pd.DataFrame(_linhas).T
print(_T.round(3).to_string())

Candidato escolhido por regiao, nas dobras e sementes:

  oven_hmi       rf|agrupado: 51/51
  ct_knobs       rf|detalhado: 35/51  ;  rf|agrupado: 13/51  ;  rf+gb|agrupado: 2/51  ;  rf+gb|detalhado: 1/51
  main_vent_sys  et+gb|detalhado: 51/51
  ipc_cooling    rf|agrupado: 48/51  ;  rf|detalhado: 3/51
  mainback       rf|detalhado: 51/51
  ipc_comp       rf|agrupado: 49/51  ;  rf+gb|agrupado: 2/51
  coil_temps     rf+gb|detalhado: 47/51  ;  rf+gb|agrupado: 4/51

Regra dominante, que e a que a entrega adota:
  oven_hmi        rf     perdas agrupado
  ct_knobs        rf     perdas detalhado
  main_vent_sys   et+gb  perdas detalhado
  ipc_cooling     rf     perdas agrupado
  mainback        rf     perdas detalhado
  ipc_comp        rf     perdas agrupado
  coil_temps      rf+gb  perdas detalhado

Erro medio absoluto de cada candidato, por regiao, para leitura do porque:

regiao                    rf detalhado           rf agrupado       rf+gb detalhado        rf+gb agrupado       et+gb det

### Leitura

A escolha por região rende 0,146 grau sobre a formulação anterior, que já escolhia o estimador por região com `losses` no tratamento detalhado, com intervalo de confiança entre 0,062 e 0,233 grau, calculado por leitura e por isso mais estreito do que deveria (ver nota abaixo). O teste de Wilcoxon não acompanha o intervalo com a mesma força, com p de 0,077, acima do nível usual de cinco por cento, de modo que o ganho é consistente na média, mas não se distribui uniformemente entre as leituras.

A escolha é unânime em três regiões, Oven HMI, Main Vent Sys e Mainback, com cinquenta e uma seleções em cinquenta e uma, e quase unânime em outras três, com quarenta e sete a quarenta e nove. A exceção é CT Knobs, em que o Random Forest com `losses` no tratamento detalhado vence em trinta e cinco escolhas e o Random Forest com `losses` no tratamento agrupado, em treze: ali os candidatos quase empatam, a 2,952 e 2,957 graus, e a escolha oscila sem custo relevante. É a condição sob a qual a lição das seções 21 e 22 diz que a seleção se sustenta: o que decide não é a largura da grade e sim a separação entre os candidatos.

O movimento de maior peso está em IPC Comp, que é também a região de maior massa de erro redutível. Ali a seleção passa `losses` ao tratamento agrupado e troca o estimador, da média entre Random Forest e Gradient Boosting para o Random Forest puro, e a região passa de 7,027 para 6,677 graus, numa região que responde por metade das leituras do projeto. O segundo movimento está em Main Vent Sys, em que a média entre Extra Trees e Gradient Boosting é escolhida em todas as dobras e sementes e a região passa de 3,890 para 3,739 graus.

Quanto à triagem de segurança, a formulação adotada registra 23,3 aprovações indevidas, 157 reprovações desnecessárias e detecção de 87,9 por cento. A troca em Main Vent Sys não altera esses números, porque a região não tem leituras acima do limite.

Registre-se, por fim, que esta seção não toca o conjunto de teste reservado. A escolha da regra foi feita inteiramente sobre a validação cruzada.


> **Onde estão as seções 21 e 22.** O texto acima remete a elas; são seções de
> `notebooks/colabs_S4/modelagem.ipynb`, e não deste notebook, que reproduz apenas a
> seção 23.
>
> **Nota sobre o intervalo de confiança acima.** O intervalo foi calculado reamostrando
> **leituras**, e não configurações. Como as leituras de um mesmo ensaio não são independentes,
> ele sai mais estreito do que deveria, e por isso não deve ser lido como prova do ganho. A
> leitura conservadora é a do teste de Wilcoxon da mesma célula, com p de 0,077, que **não**
> atinge o nível usual de 5%. O bootstrap por configuração, que é o método que o projeto adota
> em `analises_complementares.ipynb`, está registrado como pendência: **a célula de código
> da validação cruzada**, acima, é cópia fiel da Seção 23 de `modelagem.ipynb`, e alterá-la
> aqui desfaria essa correspondência.

### Avaliação no conjunto de teste reservado

A regra dominante da seleção é aplicada uma única vez aos dez ensaios reservados, com os sete modelos ajustados sobre a totalidade dos quarenta e seis ensaios de desenvolvimento, que é a condição do modelo efetivamente entregue.

Uma ressalva precede o número e não deve ser omitida de nenhum relato que o cite. Este conjunto já foi lido diversas vezes ao longo do projeto, e cada leitura consome parte da proteção que ele oferece. Além disso, a sua composição é desequilibrada: o tipo de ensaio broil c1 não tem representação alguma nele, e o tipo pyro, que é o regime mais quente e o que mais produz excessos de limite, comparece com um único ensaio entre os oito existentes.

Por essas duas razões, a documentação do projeto registra desde a terceira sprint que a leitura conservadora do desempenho é a da validação cruzada, que cobre os dois regimes. Essa orientação foi escrita quando o conjunto reservado favorecia o modelo, e é mantida agora.

In [4]:
# A regra e escrita como constante, e nao lida de _DOMINANTE, para que a entrega
# nao dependa da ordem de execucao das celulas. A celula seguinte confere que as
# duas coincidem e relata divergencia em vez de aborta-la.
_REGRA = {"oven_hmi": ("rf", "agrupado"), "ct_knobs": ("rf", "detalhado"),
          "main_vent_sys": ("et+gb", "detalhado"), "ipc_cooling": ("rf", "agrupado"),
          "mainback": ("rf", "detalhado"), "ipc_comp": ("rf", "agrupado"),
          "coil_temps": ("rf+gb", "detalhado")}
_SEMENTE_ENTREGA = 42

if "_DOMINANTE" in dir():
    _divergem = [a for a in _AREAS if _DOMINANTE[a] != _REGRA[a]]
    if _divergem:
        print("Aviso: a selecao aninhada desta execucao diverge da regra escrita em:")
        for a in _divergem:
            print(f"  {a:<15} selecionado {_DOMINANTE[a]} | escrito {_REGRA[a]}")
        print()
    else:
        print("A regra escrita coincide com a selecao aninhada desta execucao.\n")


def _completo(area, tratamento):
    """Formato longo conservando os ensaios reservados, marcados a parte."""
    d, X, alvos = _entradas(area, tratamento)
    return _longo(d, X, alvos)


print(f"Ensaios de desenvolvimento: {56 - len(_TESTE)} | reservados: {len(_TESTE)}")
print("Regra aplicada:")
for a, (e, t) in _REGRA.items():
    print(f"  {a:<15} {e:<6} perdas {t}")
print()

_linhas_teste = []
for area in _AREAS:
    _est, _trat = _REGRA[area]
    Xl, yl, cl, rl, comps = _completo(area, _trat)
    tes = np.isin(rl, list(_TESTE))
    tr = ~tes
    # Patamar e escala vem apenas do desenvolvimento, como em toda a modelagem.
    pat = pd.Series(yl[tr]).groupby(cl[tr]).mean()
    dsv = pd.Series(yl[tr]).groupby(cl[tr]).std().fillna(1.0).replace(0, 1.0)
    ger = float(yl[tr].mean())
    M = lambda m: np.hstack([Xl[m],
                             pd.Series(cl[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                             pd.Series(cl[m]).map(dsv).fillna(1.0).to_numpy().reshape(-1, 1)])
    s = StandardScaler()
    Xt, Xv = s.fit_transform(M(tr)), s.transform(M(tes))
    p = _ajusta(Xt, yl[tr], Xv, _est, _SEMENTE_ENTREGA,
                _peso_perto_do_limite(yl[tr], [comps[i] for i in np.where(tr)[0]]))
    for i, idx in enumerate(np.where(tes)[0]):
        _linhas_teste.append({"area": area, "componente": comps[idx], "real": yl[idx],
                              "pred": p[i]})

_T = pd.DataFrame(_linhas_teste)
_T["dif"] = _T["pred"] - _T["real"]
_T["limite"] = _T["componente"].map(_LIM)

print("Desempenho no conjunto de teste reservado, por area:\n")
_saida = {}
for area in _AREAS + ["AGREGADO"]:
    g = _T if area == "AGREGADO" else _T[_T["area"] == area]
    d, r = g["dif"].to_numpy(), g["real"].to_numpy()
    _saida[area] = {"n": len(g), "MAE": np.abs(d).mean(), "RMSE": float(np.sqrt((d ** 2).mean())),
                    "R2": float(1 - (d ** 2).sum() / ((r - r.mean()) ** 2).sum()),
                    "vies": d.mean(), "MAPE_%": float((np.abs(d) / r).mean() * 100),
                    "dentro_10_%": float((np.abs(d) / r < 0.10).mean() * 100)}
print(pd.DataFrame(_saida).T.round(3).to_string())

_L = _T.dropna(subset=["limite"]).copy()
_L["estoura"] = _L["real"] >= _L["limite"]
print(f"\nLeituras com limite: {len(_L)} | que atingem o limite: {int(_L['estoura'].sum())}\n")
print(f"{'margem':>7}{'falsos aprovados':>19}{'falsos reprovados':>19}{'deteccao':>11}")
for m in (0, 5, 8, 10, 12):
    alarme = _L["pred"] + m >= _L["limite"]
    fa = int((_L["estoura"] & ~alarme).sum()); fr = int((~_L["estoura"] & alarme).sum())
    det = 100 * (_L["estoura"] & alarme).sum() / _L["estoura"].sum()
    print(f"{m:>7}{fa:>19}{fr:>19}{det:>10.1f}%")

print("\nReferencias documentadas, no mesmo conjunto:")
print("  Develop, perfil e motivo em seis indicadores, floresta:   MAE 4,37 | dentro de 10%: 85,7%")
print("  S4 antes da escolha de estimador, floresta em todas:      MAE 5,15 | dentro de 10%: 83,9%")
print("  S4 com escolha de estimador e perdas detalhadas:          MAE 5,17 | dentro de 10%: 82,0%")

A regra escrita coincide com a selecao aninhada desta execucao.

Ensaios de desenvolvimento: 46 | reservados: 10
Regra aplicada:
  oven_hmi        rf     perdas agrupado
  ct_knobs        rf     perdas detalhado
  main_vent_sys   et+gb  perdas detalhado
  ipc_cooling     rf     perdas agrupado
  mainback        rf     perdas detalhado
  ipc_comp        rf     perdas agrupado
  coil_temps      rf+gb  perdas detalhado

Desempenho no conjunto de teste reservado, por area:

                   n    MAE    RMSE     R2   vies  MAPE_%  dentro_10_%
oven_hmi        30.0  2.126   2.930  0.585 -0.878   3.522       96.667
ct_knobs        60.0  2.267   3.264  0.800 -0.961   4.288       88.333
main_vent_sys   50.0  3.708   5.862  0.922  0.789   4.624       88.000
ipc_cooling     39.0  3.065   4.842  0.946 -1.494   4.615       87.179
mainback        99.0  4.458   8.902  0.879 -3.470   5.873       83.838
ipc_comp       363.0  5.989   8.590  0.700  0.018   6.628       78.237
coil_temps      76.0  7.576 

### Leitura do conjunto reservado

A formulação entregue, já com o peso próximo ao limite e com a média entre Extra Trees e Gradient Boosting em Main Vent Sys, mede 5,154 graus no conjunto reservado, contra 5,17 da formulação que escolhia apenas o estimador e 5,15 da formulação de referência. A divergência com a versão anterior do projeto, que mede 4,37 ali, permanece, e diminuiu pouco, de 0,80 para 0,78 grau. A fração de leituras dentro de dez por cento passou de 82,0 para 82,6 por cento, ainda abaixo dos 83,9 da formulação de referência. Na triagem com margem de oito graus, das 63 leituras que atingem o limite, 12 passam sem alarme, com 47 alarmes desnecessários e detecção de 81,0 por cento.

No conjunto reservado, a diferença para as duas formulações anteriores é de dois a três centésimos de grau, o que equivale a um empate. O resultado não confirma o ganho medido na validação cruzada, mas também não o contradiz: ao contrário das formulações testadas por busca de hiperparâmetros, que venciam na validação e perdiam no reservado, sinal de ajuste às dobras, esta não perde no erro médio em nenhum dos dois. Cabe registrar que duas mudanças que melhoraram a validação cruzada pioraram o conjunto reservado. O peso próximo ao limite levou o erro de 4,896 para 5,143 graus. A troca em Main Vent Sys, que reduz o erro da região de 3,890 para 3,739 graus na validação, eleva o erro da região no conjunto reservado de 3,538 para 3,708, com cinquenta leituras. A escolha de manter as duas seguiu a leitura conservadora descrita abaixo.

Ainda assim, o confronto entre as duas leituras não está resolvido, e a recomendação do projeto continua a ser a de tratar a validação cruzada como leitura conservadora, por cobrir os dois regimes de ensaio.

O caminho para resolver a divergência não é estatístico e sim experimental: são necessários ensaios de broil c1, hoje ausentes do conjunto reservado, e mais ensaios de pyro, hoje representados por um único caso. Enquanto eles não existirem, qualquer conclusão sobre qual das duas leituras descreve melhor o uso real permanecerá sem sustentação.

> **Duas notas sobre o texto acima**, que vem da Seção 23 de `modelagem.ipynb`, aqui com os
> nomes técnicos no original em inglês.
>
> **O que é a "versão anterior do projeto".** É o modelo da Sprint 3, registrado na seção 4.3
> da documentação, e é dele que vêm os 4,37 graus. Ele foi medido na **base anterior**, que
> tinha o sensor duplicado do Oven HMI, 74 componentes e 727 previsões de teste, contra 73 e
> 717 de hoje. A seção 5 de `analises_complementares.ipynb` mostra que contar o mesmo sensor
> duas vezes fazia aquela base parecer melhor, de modo que parte da diferença para os 5,154 vem
> da base, e não do modelo. A comparação não é direta. Pelo mesmo motivo, o rótulo `Develop` na
> saída da célula anterior nomeia uma branch, e não um modelo.
>
> **Os 5,143 graus citados no texto.** Referem-se à formulação anterior à troca do estimador em
> Main Vent Sys. Com essa troca, que é a da entrega, o valor passa a ser os **5,154** impressos
> pela célula acima.

## Exemplo de triagem

In [5]:
# Exemplo de uso: a triagem que o modelo sustenta. Entre os ensaios reservados, as dez
# leituras de menor folga prevista, que sao as que a engenharia examinaria primeiro.
# Margem de oito graus: criterio adotado para comparar os modelos entre si (documentacao,
# Secao 4.4). Em producao vale a regra de projeto da Whirlpool, que exige folga de pelo
# menos 10 por cento do limite (FOLGA_ATENCAO_PCT no src/frontend/dados/exportar.py).
MARGEM = 8.0

_D = _T.dropna(subset=["limite"]).copy()
_D["folga_prevista"] = _D["limite"] - _D["pred"]
_D["sinalizado"] = _D["pred"] + MARGEM >= _D["limite"]
_D["estourou"] = _D["real"] >= _D["limite"]

print(f"Triagem com margem de {MARGEM:.0f} graus, nos {len(_D)} pares com limite tabelado.")
print(f"Sinalizados: {int(_D['sinalizado'].sum())} | que de fato atingem o limite: "
      f"{int(_D['estourou'].sum())}\n")
print("Dez leituras de menor folga prevista:\n")
_cols = ["area", "componente", "real", "pred", "limite", "folga_prevista", "sinalizado", "estourou"]
print(_D.nsmallest(10, "folga_prevista")[_cols].round(2).to_string(index=False))

Triagem com margem de 8 graus, nos 663 pares com limite tabelado.
Sinalizados: 98 | que de fato atingem o limite: 63

Dez leituras de menor folga prevista:

    area componente   real   pred  limite  folga_prevista  sinalizado  estourou
ipc_comp    IGBT 11 120.92 126.72   110.0          -16.72        True      True
ipc_comp    IGBT 10 120.72 126.65   110.0          -16.65        True      True
ipc_comp    IGBT 12 120.76 126.56   110.0          -16.56        True      True
ipc_comp     IGBT 8 120.64 125.91   110.0          -15.91        True      True
ipc_comp     IGBT 9 120.26 125.48   110.0          -15.48        True      True
ipc_comp     IGBT 7 120.30 125.38   110.0          -15.38        True      True
ipc_comp     IGBT 6 119.62 124.85   110.0          -14.85        True      True
ipc_comp     IGBT 5 118.69 123.18   110.0          -13.18        True      True
ipc_comp    IGBT 10 118.80 123.03   110.0          -13.03        True      True
ipc_comp    IGBT 11 118.63 122.97   110.0  

### Leitura da triagem

A tabela mostra o produto como ele é usado: para uma configuração ainda não ensaiada, o modelo
ordena os componentes por folga prevista, e a engenharia dirige o ensaio físico aos primeiros.
A coluna `sinalizado` é a decisão do modelo; a coluna `estourou` é o que de fato aconteceu.

As duas colunas não coincidem sempre, e é por isso que a entrega é **triagem e não substituição
do ensaio**. O erro caro é a linha em que `estourou` é verdadeiro e `sinalizado` é falso, isto
é, o componente que passou sem alarme. A margem de oito graus existe para reduzir esse caso, ao
custo de alarmes desnecessários, que custam um reensaio.

Duas ressalvas sobre esta tabela. A margem de oito graus é o critério adotado para comparar
os modelos entre si; **em produção o dashboard aplica a regra de projeto da Whirlpool**, que
sinaliza o componente quando a folga fica abaixo de 10% do limite. E as dez linhas vêm de
**ensaios reservados diferentes**, reunidas por folga prevista, e não de uma única
configuração: é por isso que o mesmo componente aparece mais de uma vez. Nas dez, a
sinalização acertou; os 12 casos em que o componente estourou sem alarme estão contados na
linha de margem 8 da célula anterior.

## Conclusão

O modelo entregue mede entre **5,96 e 6,23 graus** de erro absoluto médio na validação cruzada,
conforme o método de aninhamento, contra 6,107 da formulação que escolhia apenas o estimador,
6,47 da formulação de referência, que usa Random Forest em todas as áreas (seção 19 de
`modelagem.ipynb`), e 6,565 do modelo da Sprint 3, este medido na base anterior. No conjunto de teste reservado mede
**5,154 graus**, contra 5,17 da formulação que escolhia apenas o estimador, 5,15 da formulação
de referência (seção 20) e 4,37 do modelo da Sprint 3.

Duas ressalvas acompanham o número e não devem ser separadas dele. A primeira é que o conjunto
reservado **aponta em sentido contrário ao da validação**: ali o modelo da Sprint 3 mede 4,37 graus, melhor que a
atual, ressalvado que aquele número vem da base anterior, como a nota acima registra. A composição desse conjunto é desequilibrada, como a seção
"Avaliação no conjunto de teste reservado" registra acima, e a leitura conservadora adotada pelo projeto desde a terceira sprint é a da
validação cruzada, que cobre os dois regimes de ensaio. A segunda é que a regra por área foi
escolhida observando os resultados. Escrita como constante, ela mede 5,855 graus, conforme a
seção 9 de `comparacao_modelos_final.ipynb`; com a escolha refeita dentro de cada dobra, mede de
5,96 a 6,23 graus, e é essa faixa, mais conservadora, que o projeto reporta.

O modelo é levado a produção por `src/frontend/dados/exportar.py`, que executa esta mesma seção e
grava `src/frontend/dados/modelos/modelos.joblib`, consultado pelo dashboard na previsão sob
demanda.

## Referências

BREIMAN, Leo. Random forests. **Machine Learning**, v. 45, n. 1, p. 5-32, 2001.

FRIEDMAN, Jerome H. Greedy function approximation: a gradient boosting machine. **The Annals
of Statistics**, v. 29, n. 5, p. 1189-1232, 2001.

GEURTS, Pierre; ERNST, Damien; WEHENKEL, Louis. Extremely randomized trees. **Machine
Learning**, v. 63, n. 1, p. 3-42, 2006.

KAUFMAN, Shachar; ROSSET, Saharon; PERLICH, Claudia; STITELMAN, Ori. Leakage in data mining:
formulation, detection, and avoidance. **ACM Transactions on Knowledge Discovery from Data**,
v. 6, n. 4, artigo 15, 2012.

PEDREGOSA, Fabian et al. Scikit-learn: machine learning in Python. **Journal of Machine
Learning Research**, v. 12, p. 2825-2830, 2011.